In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

client = OpenAI()

In [4]:
# Load the word probabilities
df_probs = pd.read_csv(r'C:\Users\jonat\Lasso_paper\Narratives\narratives_construction\_data\word_probabilities.csv', index_col=0)

# 1. DROP META-TOPICS: Remove topics that describe academic writing rather than economic variables
meta_topics_to_drop = ['Research', 'Humor/language', 'Corrections/amplifications', 'Problems']
cols_to_keep = [col for col in df_probs.columns if col not in meta_topics_to_drop]
df_probs = df_probs[cols_to_keep]

topic_definitions = {}
for topic in df_probs.columns:
    # 2. SAVE TOKENS: Grab only the top 5 terms instead of 10. 
    # This cuts the system prompt size in half while retaining the core semantic meaning.
    top_5_terms = df_probs[topic].sort_values(ascending=False).head(5).index.tolist()
    topic_definitions[topic] = top_5_terms

# Format this into an LLM-friendly string
topics_formatted_list = [f"- **{topic}**: {', '.join(words)}" for topic, words in topic_definitions.items()]
topics_context_string = "\n".join(topics_formatted_list)

print("Extracted Topics Context (Preview):")
print("\n".join(topics_formatted_list[:5]))

Extracted Topics Context (Preview):
- **Natural disasters**: water, area, damage, people, storm
- **Internet**: internet, site, online, web, service
- **Soft drinks**: company, market, america, north, american
- **Mobile devices**: apple, phone, device, company, mobile
- **Profits**: profit, sale, rose, net, earlier


In [ ]:
def evaluate_paper_claim(metadata_dict, abstract_text, topics_context, model="gpt-4o-mini"):
    """
    Prompts the LLM to determine if the abstract claims a topic predicts stock returns.
    Optimized for token efficiency and preventing meta-language hallucinations.
    """
    

    system_prompt = f"""You are an expert financial economist. 
Your task is to determine if an academic paper's abstract claims that a specific variable PREDICTS stock returns (aggregate or cross-sectional).

Permitted topics and their top keywords:
{topics_context}

CRITICAL RULES:
1. DISTINGUISH METHODOLOGY FROM SUBJECT: Ignore standard academic meta-language (e.g., "our study shows", "we analyze data", "the results suggest"). Focus ONLY on the actual economic or financial variable being tested.
2. If a claim is made, classify it into EXACTLY ONE of the permitted topics. Do not invent topics.
3. Output strictly in JSON format with exactly these two keys:
   - "predicts_returns": boolean (true if a predictive claim regarding stock returns is made, false otherwise)
   - "topic": string (the exact name of the topic from the list, or null if false)
"""

    user_prompt = f"""
Title: {metadata_dict.get('title', 'N/A')}
Abstract: {abstract_text}
"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={ "type": "json_object" },
            temperature=0.0 
        )
        
        return json.loads(response.choices[0].message.content)
        
    except Exception as e:
        # Return a clean fallback to prevent pipeline crashes
        return {"predicts_returns": False, "topic": None}

In [ ]:
# Load the papers outputted by your get_abstracts pipeline
df_papers = pd.read_csv(r'C:\Users\jonat\Lasso_paper\Narratives\openalex_abstracts_out\abstracts.csv')

# Filter only papers where an abstract was successfully found
df_valid_papers = df_papers[df_papers['has_abstract'] == True].copy()
print(f"Found {len(df_valid_papers)} papers with abstracts to process.")

def process_row(row):
    metadata = {
        "title": row.get("title"),
    }
    # Removed venue, date, and authors from being sent to the LLM to save input tokens.
    # The LLM only needs the title and abstract to determine the predictive claim.
    abstract = row.get("abstract")
    
    llm_result = evaluate_paper_claim(metadata, abstract, topics_context_string)
    
    return {
        "index": row.name, 
        "claims_predicts_returns": llm_result.get("predicts_returns", False),
        "predicted_topic": llm_result.get("topic", None)
    }

MAX_WORKERS = 20
results = []

print(f"Kicking off {MAX_WORKERS} concurrent workers...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_row = {executor.submit(process_row, row): row for _, row in df_valid_papers.iterrows()}
    
    for future in tqdm(as_completed(future_to_row), total=len(df_valid_papers)):
        try:
            results.append(future.result())
        except Exception as exc:
            pass # Silently handle to keep the progress bar clean, or print if debugging

# Merge results
results_df = pd.DataFrame(results).set_index("index")
df_valid_papers = df_valid_papers.join(results_df)

output_file = 'papers_with_claims_extracted.csv'
df_valid_papers.to_csv(output_file, index=False)
print(f"Extraction complete. Saved results to {output_file}")

In [ ]:
# Filter the dataframe to see only the positive hits
df_hits = df_valid_papers[df_valid_papers['claims_predicts_returns'] == True]

print(f"Total papers claiming predictive power: {len(df_hits)}")

# Display the title and the assigned topic
display(df_hits[['title', 'predicted_topic']].head(10))